In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp
import emusetup_and_mc as emc   # shared emulator build / training and Monte Carlo IC helpers
import obs as obs

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
# using mid

Idea with this notebook is to use ablative methods to test parameter importance. In other words, train multiple models on different combinations of the following parameters: (X, Y, Z, (ecc, prec, obl)). In this sense, we can test the quality of different dynamical setups to SM90 (i.e. can the emulator learn a dynamical system with reduced variables). To reduce computational time, only the following combinations will be used:

1. Full setup w/ decomposed insolation: (X, Y, Z, (ecc, prec, obl))
2. State variables only (no forcing): (X, Y, Z)
3. Drop one state variable: (X, Y,, (ecc, prec, obl)); (X,, Z, (ecc, prec, obl)); (X,,, (ecc, prec, obl))
4. Forcing only (no state variables): (,(ecc, prec, obl)) ~ this is essentially a regression right?
5. Full setup with original insolation (control) ~ already done I guess, so no need to do this?

### Experimental Prep

In [ ]:
DATASET = rc.DATASET        # '1461ka'
TRAIN_START = 'mid'         # key of rc.STARTS: 'early', 'mid', 'late', 'balanced' (1461-579 ka, no test set) or 'default' (80:20 split)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set

STUDY_DIR = rc.RUNS_DIR / '_Ablations'   # every ablation run goes in Scripts/runs/_Ablations/<run_tag>/
STUDY_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- phase space with all three pairs (X-Y, X-Z, Y-Z), also for setups that drop a state variable ----
# The ground truth (GP mean of X, Y, Z) is drawn in every panel; the rollout only in the panels whose two variables
# are both in the setup's state. Each column's scaling depends only on that column's training rows, which are the
# same in every setup, so the ground truth here is on the same scale as every setup's rollout
_cfg_xyz = rc.make_config(DATASET, seed=42, params=PARAMS, train_start=TRAIN_START)
_splits_xyz = rc.split_data(rc.load_data(_cfg_xyz), _cfg_xyz)
GT_XYZ = rc.fit_scaler(_splits_xyz).transform(_splits_xyz['full'])[:, :3]   # scaled X, Y, Z ground truth

def phase_space_xyz(cfg, rollout_data, title=None, colors=sp.DEFAULT_COLORS):
    """
        cfg          - the setup's config (which state variables its rollout has, in which columns)
        rollout_data - the setup's full-record rollout (states first, then forcings)
    """
    names = rc.state_names(cfg)
    pairs = [('X', 'Y'), ('X', 'Z'), ('Y', 'Z')]
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    n = rollout_data.shape[0]
    for k, (ax, (a, b)) in enumerate(zip(axs, pairs)):
        ia, ib = 'XYZ'.index(a), 'XYZ'.index(b)
        ax.plot(GT_XYZ[:n, ia], GT_XYZ[:n, ib], color=colors[k][0], linestyle='dashed', label="scaled data")
        if a in names and b in names:
            ax.plot(rollout_data[:, names.index(a)], rollout_data[:, names.index(b)], color=colors[k][1],
                    label="emulator data")
        else:
            missing = [v for v in (a, b) if v not in names]
            ax.set_title(f"{' and '.join(missing)} not in this setup's state: ground truth only", fontsize=11)
        ax.set_xlabel(a)
        ax.set_ylabel(b)
    if title:
        fig.suptitle(title, fontweight='bold')
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.005), ncol=3, fontsize=20,
               frameon=True, facecolor='white', edgecolor='gray')
    plt.show()

In [ ]:
# ---- Monte Carlo settings shared by every setup: the model seeds and the GP-drawn initial conditions ----
# Every setup that runs a Monte Carlo ensemble trains one model per seed in MC_SEEDS and rolls each model out from
# the same N_IC initial conditions, so the setups are compared on identical draws.
# emc.generate_IC saves the draws to study_dir (this notebook's STUDY_DIR) the first time and loads them from there
# afterwards: obs.gp_reg fits each GP with random optimizer restarts, so refitting in a new session can give slightly
# different GPs, and with them different draws. emc.make_ic lays them out and scales them for one setup
MC_SEEDS = list(range(30))     # one model per seed
N_IC = 100                     # initial conditions drawn from the GP of each state record
IC_RANDOM_STATE = 42           # draws of record k use IC_RANDOM_STATE + k (independent draws for X, Y, Z)
N_WORKERS = 4                  # parallel training processes (emc.train_mc)
N_THREADS = 2                  # TensorFlow threads per process

IC_DRAWS = emc.generate_IC(N_IC, PARAMS, train_start=TRAIN_START, ic_randomstate=IC_RANDOM_STATE,
                             study_dir=STUDY_DIR)   # (N_IC, n_ages, 3)

### Setup 0 -- full setup w/ ber90 insolation (control), Monte Carlo
One model per seed in MC_SEEDS (30) is trained on the GP mean record; every model is rolled out from the same N_IC (100)
initial conditions drawn from the GP (both set in the experimental prep section). Reported: the L2 of each model's mean
rollout (over its ICs) against the GP mean, whole record and test set, and its mean and variance over the models.

In [ ]:
# config 0 (control): X, Y, Z with the Berger 1990 65N June-solstice insolation as the single forcing - the
# standard setup, run through the same pipeline as the ablations so it compares directly with setups 1-5
# cfg0 describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG0_KW = dict(train_start=TRAIN_START, insol='insol_ber90', forcing=('insol',))
cfg0 = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG0_KW)
params0 = cfg0['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC0_SEEDS = MC_SEEDS            # one model per seed
N_IC0 = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC0_DIR = STUDY_DIR / 'setup0_MC'   # runs/_Ablations/setup0_MC/<run_tag per seed>/
RETRAIN0 = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC0_PATHS = emc.mc_paths(DATASET, CFG0_KW, MC0_SEEDS, MC0_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg0['run_tag'], "| inputs:", rc.inputs_label(cfg0), "| seeds:", MC0_SEEDS[0], "-", MC0_SEEDS[-1], "| ICs:", N_IC0)
print("params  :", params0)

In [ ]:
data0 = LPNE.prepare_data(cfg0)
splits0, scaler0 = data0['splits'], data0['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC0 = emc.make_ic(cfg0, N_IC0, PARAMS, scaler0, splits0, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC0.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC0_BEST_EPOCH, MC0_HIST = emc.train_mc(DATASET, CFG0_KW, MC0_SEEDS, MC0_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN0)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL0[model, IC] (scaled, states + forcings) ----
w0 = cfg0['window_size']
emu0, ROLL0 = emc.rollout_mc(cfg0, PARAMS, MC0_PATHS, IC0)

full0 = scaler0.transform(splits0['full'])   # ground truth: the GP mean record, scaled
fullt0 = splits0['time']
testt0, traint0 = splits0['test_time'], splits0['train_time']
test_end_0 = (testt0.min() + traint0.max()) / 2   # test | train boundary
MEAN_ROLL0 = ROLL0.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_0 = MEAN_ROLL0.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL0.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC0 ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC0_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC0_TABLE, MC0_SUMMARY = emc.mc_l2(cfg0, full0, ROLL0, MC0_SEEDS)
#display(MC0_SUMMARY.style.format('{:.4f}'))
#display(MC0_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg0, ROLL0, full0, fullt0, MC0_SUMMARY, test_end_0,
                     title=f"setup 0 ({rc.inputs_label(cfg0)})")

In [ ]:
# phase space (X-Y, X-Z, Y-Z) of setup 0's ensemble-mean rollout (mean over all models and ICs) over the ground truth
sp.phase_space(emu0, full0, w0, rollout_data=rollout_full_0)

In [ ]:
# multitaper spectrum of X (ice mass): setup 0's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_0 = sp.frequency_spectra(emu0, full0, w0,
                                  timestep=cfg0['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_0)

### Setup 1 -- full setup w/ decomposed insolation 

In [ ]:
# config 1: X, Y, Z with the decomposed insolation (eccentricity, precession index, obliquity)
# cfg1 describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG1_KW = dict(train_start=TRAIN_START, forcing=('ecc', 'prec', 'obl'))
cfg1 = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG1_KW)
params1 = cfg1['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC1_SEEDS = MC_SEEDS            # one model per seed
N_IC1 = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC1_DIR = STUDY_DIR / 'setup1_MC'   # runs/_Ablations/setup1_MC/<run_tag per seed>/
RETRAIN1 = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC1_PATHS = emc.mc_paths(DATASET, CFG1_KW, MC1_SEEDS, MC1_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg1['run_tag'], "| inputs:", rc.inputs_label(cfg1), "| seeds:", MC1_SEEDS[0], "-", MC1_SEEDS[-1], "| ICs:", N_IC1)
print("params  :", params1)

In [ ]:
data1 = LPNE.prepare_data(cfg1)
splits1, scaler1 = data1['splits'], data1['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC1 = emc.make_ic(cfg1, N_IC1, PARAMS, scaler1, splits1, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC1.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC1_BEST_EPOCH, MC1_HIST = emc.train_mc(DATASET, CFG1_KW, MC1_SEEDS, MC1_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN1)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL1[model, IC] (scaled, states + forcings) ----
w1 = cfg1['window_size']
emu1, ROLL1 = emc.rollout_mc(cfg1, PARAMS, MC1_PATHS, IC1)

full1 = scaler1.transform(splits1['full'])   # ground truth: the GP mean record, scaled
fullt1 = splits1['time']
testt1, traint1 = splits1['test_time'], splits1['train_time']
test_end_1 = (testt1.min() + traint1.max()) / 2   # test | train boundary
MEAN_ROLL1 = ROLL1.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_1 = MEAN_ROLL1.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL1.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC1 ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC1_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC1_TABLE, MC1_SUMMARY = emc.mc_l2(cfg1, full1, ROLL1, MC1_SEEDS)
#display(MC1_SUMMARY.style.format('{:.4f}'))
#display(MC1_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg1, ROLL1, full1, fullt1, MC1_SUMMARY, test_end_1,
                     title=f"setup 1 ({rc.inputs_label(cfg1)})")

In [ ]:
# phase space (X-Y, X-Z, Y-Z) of setup 1's ensemble-mean rollout (mean over all models and ICs) over the ground truth
sp.phase_space(emu1, full1, w1, rollout_data=rollout_full_1)

In [ ]:
# multitaper spectrum of X (ice mass): setup 1's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_1 = sp.frequency_spectra(emu1, full1, w1,
                                  timestep=cfg1['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_1)

### Setup 2 -- state variables only (with no forcing)

In [ ]:
# config 2: X, Y, Z with no forcing
# cfg2 describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG2_KW = dict(train_start=TRAIN_START, forcing=())
cfg2 = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG2_KW)
params2 = cfg2['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC2_SEEDS = MC_SEEDS            # one model per seed
N_IC2 = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC2_DIR = STUDY_DIR / 'setup2_MC'   # runs/_Ablations/setup2_MC/<run_tag per seed>/
RETRAIN2 = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC2_PATHS = emc.mc_paths(DATASET, CFG2_KW, MC2_SEEDS, MC2_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg2['run_tag'], "| inputs:", rc.inputs_label(cfg2), "| seeds:", MC2_SEEDS[0], "-", MC2_SEEDS[-1], "| ICs:", N_IC2)
print("params  :", params2)

In [ ]:
data2 = LPNE.prepare_data(cfg2)
splits2, scaler2 = data2['splits'], data2['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC2 = emc.make_ic(cfg2, N_IC2, PARAMS, scaler2, splits2, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC2.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC2_BEST_EPOCH, MC2_HIST = emc.train_mc(DATASET, CFG2_KW, MC2_SEEDS, MC2_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN2)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL2[model, IC] (scaled, states + forcings) ----
w2 = cfg2['window_size']
emu2, ROLL2 = emc.rollout_mc(cfg2, PARAMS, MC2_PATHS, IC2)

full2 = scaler2.transform(splits2['full'])   # ground truth: the GP mean record, scaled
fullt2 = splits2['time']
testt2, traint2 = splits2['test_time'], splits2['train_time']
test_end_2 = (testt2.min() + traint2.max()) / 2   # test | train boundary
MEAN_ROLL2 = ROLL2.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_2 = MEAN_ROLL2.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL2.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC2 ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC2_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC2_TABLE, MC2_SUMMARY = emc.mc_l2(cfg2, full2, ROLL2, MC2_SEEDS)
#display(MC2_SUMMARY.style.format('{:.4f}'))
#display(MC2_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg2, ROLL2, full2, fullt2, MC2_SUMMARY, test_end_2,
                     title=f"setup 2 ({rc.inputs_label(cfg2)})")

In [ ]:
# phase space (X-Y, X-Z, Y-Z) of setup 2's ensemble-mean rollout (mean over all models and ICs) over the ground truth
sp.phase_space(emu2, full2, w2, rollout_data=rollout_full_2)

In [ ]:
# multitaper spectrum of X (ice mass): setup 2's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_2 = sp.frequency_spectra(emu2, full2, w2,
                                  timestep=cfg2['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_2)

### Setup 3a -- (X,Y, ,(decomposed insol))

In [ ]:
# config 3a: X, Y with the decomposed insolation (Z dropped)
# cfg3a describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG3a_KW = dict(train_start=TRAIN_START, state=('X', 'Y'), forcing=('ecc', 'prec', 'obl'))
cfg3a = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG3a_KW)
params3a = cfg3a['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC3a_SEEDS = MC_SEEDS            # one model per seed
N_IC3a = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC3a_DIR = STUDY_DIR / 'setup3a_MC'   # runs/_Ablations/setup3a_MC/<run_tag per seed>/
RETRAIN3a = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC3a_PATHS = emc.mc_paths(DATASET, CFG3a_KW, MC3a_SEEDS, MC3a_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg3a['run_tag'], "| inputs:", rc.inputs_label(cfg3a), "| seeds:", MC3a_SEEDS[0], "-", MC3a_SEEDS[-1], "| ICs:", N_IC3a)
print("params  :", params3a)

In [ ]:
data3a = LPNE.prepare_data(cfg3a)
splits3a, scaler3a = data3a['splits'], data3a['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC3a = emc.make_ic(cfg3a, N_IC3a, PARAMS, scaler3a, splits3a, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC3a.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC3a_BEST_EPOCH, MC3a_HIST = emc.train_mc(DATASET, CFG3a_KW, MC3a_SEEDS, MC3a_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN3a)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL3a[model, IC] (scaled, states + forcings) ----
w3a = cfg3a['window_size']
emu3a, ROLL3a = emc.rollout_mc(cfg3a, PARAMS, MC3a_PATHS, IC3a)

full3a = scaler3a.transform(splits3a['full'])   # ground truth: the GP mean record, scaled
fullt3a = splits3a['time']
testt3a, traint3a = splits3a['test_time'], splits3a['train_time']
test_end_3a = (testt3a.min() + traint3a.max()) / 2   # test | train boundary
MEAN_ROLL3a = ROLL3a.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_3a = MEAN_ROLL3a.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL3a.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC3a ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC3a_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC3a_TABLE, MC3a_SUMMARY = emc.mc_l2(cfg3a, full3a, ROLL3a, MC3a_SEEDS)
#display(MC3a_SUMMARY.style.format('{:.4f}'))
#display(MC3a_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg3a, ROLL3a, full3a, fullt3a, MC3a_SUMMARY, test_end_3a,
                     title=f"setup 3a ({rc.inputs_label(cfg3a)})")

In [ ]:
# phase space of setup 3a's ensemble-mean rollout (mean over all models and ICs) over the ground truth, all three
# pairs (X-Y, X-Z, Y-Z): the rollout appears only in the panels whose variables are both in this setup's state
phase_space_xyz(cfg3a, rollout_full_3a, title=f"setup 3a ({rc.inputs_label(cfg3a)})")

In [ ]:
# multitaper spectrum of X (ice mass): setup 3a's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_3a = sp.frequency_spectra(emu3a, full3a, w3a,
                                  timestep=cfg3a['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_3a)

### Setup 3b -- (X, , Z,(decomposed insol))

In [ ]:
# config 3b: X, Z with the decomposed insolation (Y dropped)
# cfg3b describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG3b_KW = dict(train_start=TRAIN_START, state=('X', 'Z'), forcing=('ecc', 'prec', 'obl'))
cfg3b = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG3b_KW)
params3b = cfg3b['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC3b_SEEDS = MC_SEEDS            # one model per seed
N_IC3b = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC3b_DIR = STUDY_DIR / 'setup3b_MC'   # runs/_Ablations/setup3b_MC/<run_tag per seed>/
RETRAIN3b = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC3b_PATHS = emc.mc_paths(DATASET, CFG3b_KW, MC3b_SEEDS, MC3b_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg3b['run_tag'], "| inputs:", rc.inputs_label(cfg3b), "| seeds:", MC3b_SEEDS[0], "-", MC3b_SEEDS[-1], "| ICs:", N_IC3b)
print("params  :", params3b)

In [ ]:
data3b = LPNE.prepare_data(cfg3b)
splits3b, scaler3b = data3b['splits'], data3b['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC3b = emc.make_ic(cfg3b, N_IC3b, PARAMS, scaler3b, splits3b, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC3b.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC3b_BEST_EPOCH, MC3b_HIST = emc.train_mc(DATASET, CFG3b_KW, MC3b_SEEDS, MC3b_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN3b)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL3b[model, IC] (scaled, states + forcings) ----
w3b = cfg3b['window_size']
emu3b, ROLL3b = emc.rollout_mc(cfg3b, PARAMS, MC3b_PATHS, IC3b)

full3b = scaler3b.transform(splits3b['full'])   # ground truth: the GP mean record, scaled
fullt3b = splits3b['time']
testt3b, traint3b = splits3b['test_time'], splits3b['train_time']
test_end_3b = (testt3b.min() + traint3b.max()) / 2   # test | train boundary
MEAN_ROLL3b = ROLL3b.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_3b = MEAN_ROLL3b.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL3b.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC3b ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC3b_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC3b_TABLE, MC3b_SUMMARY = emc.mc_l2(cfg3b, full3b, ROLL3b, MC3b_SEEDS)
#display(MC3b_SUMMARY.style.format('{:.4f}'))
#display(MC3b_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg3b, ROLL3b, full3b, fullt3b, MC3b_SUMMARY, test_end_3b,
                     title=f"setup 3b ({rc.inputs_label(cfg3b)})")

In [ ]:
# phase space of setup 3b's ensemble-mean rollout (mean over all models and ICs) over the ground truth, all three
# pairs (X-Y, X-Z, Y-Z): the rollout appears only in the panels whose variables are both in this setup's state
phase_space_xyz(cfg3b, rollout_full_3b, title=f"setup 3b ({rc.inputs_label(cfg3b)})")

In [ ]:
# multitaper spectrum of X (ice mass): setup 3b's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_3b = sp.frequency_spectra(emu3b, full3b, w3b,
                                  timestep=cfg3b['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_3b)

### Setup 3c -- (X, , ,(decomposed insol))

In [ ]:
# config 3c: X only with the decomposed insolation (Y and Z dropped)
# cfg3c describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG3c_KW = dict(train_start=TRAIN_START, state=('X',), forcing=('ecc', 'prec', 'obl'))
cfg3c = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG3c_KW)
params3c = cfg3c['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC3c_SEEDS = MC_SEEDS            # one model per seed
N_IC3c = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC3c_DIR = STUDY_DIR / 'setup3c_MC'   # runs/_Ablations/setup3c_MC/<run_tag per seed>/
RETRAIN3c = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC3c_PATHS = emc.mc_paths(DATASET, CFG3c_KW, MC3c_SEEDS, MC3c_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg3c['run_tag'], "| inputs:", rc.inputs_label(cfg3c), "| seeds:", MC3c_SEEDS[0], "-", MC3c_SEEDS[-1], "| ICs:", N_IC3c)
print("params  :", params3c)

In [ ]:
data3c = LPNE.prepare_data(cfg3c)
splits3c, scaler3c = data3c['splits'], data3c['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC3c = emc.make_ic(cfg3c, N_IC3c, PARAMS, scaler3c, splits3c, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC3c.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC3c_BEST_EPOCH, MC3c_HIST = emc.train_mc(DATASET, CFG3c_KW, MC3c_SEEDS, MC3c_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN3c)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL3c[model, IC] (scaled, states + forcings) ----
w3c = cfg3c['window_size']
emu3c, ROLL3c = emc.rollout_mc(cfg3c, PARAMS, MC3c_PATHS, IC3c)

full3c = scaler3c.transform(splits3c['full'])   # ground truth: the GP mean record, scaled
fullt3c = splits3c['time']
testt3c, traint3c = splits3c['test_time'], splits3c['train_time']
test_end_3c = (testt3c.min() + traint3c.max()) / 2   # test | train boundary
MEAN_ROLL3c = ROLL3c.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_3c = MEAN_ROLL3c.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL3c.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC3c ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC3c_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC3c_TABLE, MC3c_SUMMARY = emc.mc_l2(cfg3c, full3c, ROLL3c, MC3c_SEEDS)
#display(MC3c_SUMMARY.style.format('{:.4f}'))
#display(MC3c_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg3c, ROLL3c, full3c, fullt3c, MC3c_SUMMARY, test_end_3c,
                     title=f"setup 3c ({rc.inputs_label(cfg3c)})")

In [ ]:
# phase space of setup 3c's ensemble-mean rollout (mean over all models and ICs) over the ground truth, all three
# pairs (X-Y, X-Z, Y-Z): the rollout appears only in the panels whose variables are both in this setup's state
phase_space_xyz(cfg3c, rollout_full_3c, title=f"setup 3c ({rc.inputs_label(cfg3c)})")

In [ ]:
# multitaper spectrum of X (ice mass): setup 3c's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_3c = sp.frequency_spectra(emu3c, full3c, w3c,
                                  timestep=cfg3c['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_3c)

### Setup 4 -- only insolation 

In [ ]:
# config 4: forcing only - a regression of X, Y, Z on the decomposed insolation. The model sees only the last
# window_size kyr of (ecc, prec, obl) and predicts the states directly; its own predictions are never fed back
# (state_feedback=False). Same window, split and hyperparameters as the other setups
# cfg4 describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG4_KW = dict(train_start=TRAIN_START, forcing=('ecc', 'prec', 'obl'), state_feedback=False)
cfg4 = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG4_KW)
params4 = cfg4['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC4_SEEDS = MC_SEEDS            # one model per seed
N_IC4 = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC4_DIR = STUDY_DIR / 'setup4_MC'   # runs/_Ablations/setup4_MC/<run_tag per seed>/
RETRAIN4 = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC4_PATHS = emc.mc_paths(DATASET, CFG4_KW, MC4_SEEDS, MC4_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg4['run_tag'], "| inputs:", rc.inputs_label(cfg4), "| seeds:", MC4_SEEDS[0], "-", MC4_SEEDS[-1], "| ICs:", N_IC4)
print("params  :", params4)

In [ ]:
data4 = LPNE.prepare_data(cfg4)
splits4, scaler4 = data4['splits'], data4['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC4 = emc.make_ic(cfg4, N_IC4, PARAMS, scaler4, splits4, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC4.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC4_BEST_EPOCH, MC4_HIST = emc.train_mc(DATASET, CFG4_KW, MC4_SEEDS, MC4_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN4)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL4[model, IC] (scaled, states + forcings) ----
# NOTE: the states are never fed back, so past the seed window every IC gives the same rollout - the spread
# here is between models (seeds) only
w4 = cfg4['window_size']
emu4, ROLL4 = emc.rollout_mc(cfg4, PARAMS, MC4_PATHS, IC4)

full4 = scaler4.transform(splits4['full'])   # ground truth: the GP mean record, scaled
fullt4 = splits4['time']
testt4, traint4 = splits4['test_time'], splits4['train_time']
test_end_4 = (testt4.min() + traint4.max()) / 2   # test | train boundary
MEAN_ROLL4 = ROLL4.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_4 = MEAN_ROLL4.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL4.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC4 ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC4_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC4_TABLE, MC4_SUMMARY = emc.mc_l2(cfg4, full4, ROLL4, MC4_SEEDS)
#display(MC4_SUMMARY.style.format('{:.4f}'))
#display(MC4_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg4, ROLL4, full4, fullt4, MC4_SUMMARY, test_end_4,
                     title=f"setup 4 ({rc.inputs_label(cfg4)})")

In [ ]:
# phase space (X-Y, X-Z, Y-Z) of setup 4's ensemble-mean rollout (mean over all models and ICs) over the ground truth
sp.phase_space(emu4, full4, w4, rollout_data=rollout_full_4)

In [ ]:
# multitaper spectrum of X (ice mass): setup 4's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_4 = sp.frequency_spectra(emu4, full4, w4,
                                  timestep=cfg4['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_4)

### 5 -- (X,Y,Z,(prec,,obl))

In [ ]:
# config 5: X, Y, Z with precession and obliquity only (eccentricity dropped)
# cfg5 describes the data and settings; each Monte Carlo model gets the same config with its own seed
CFG5_KW = dict(train_start=TRAIN_START, state=('X', 'Y', 'Z'), forcing=('prec', 'obl'))
cfg5 = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG5_KW)
params5 = cfg5['params']

# Monte Carlo settings: the shared seeds and initial conditions from the experimental prep section
MC5_SEEDS = MC_SEEDS            # one model per seed
N_IC5 = N_IC                    # initial conditions drawn from the GP, the same for every model and setup
MC5_DIR = STUDY_DIR / 'setup5_MC'   # runs/_Ablations/setup5_MC/<run_tag per seed>/
RETRAIN5 = False                # False: skip seeds whose weights already exist, so an interrupted run resumes
MC5_PATHS = emc.mc_paths(DATASET, CFG5_KW, MC5_SEEDS, MC5_DIR, PARAMS)   # each seed's run folder / weights

print("base run tag:", cfg5['run_tag'], "| inputs:", rc.inputs_label(cfg5), "| seeds:", MC5_SEEDS[0], "-", MC5_SEEDS[-1], "| ICs:", N_IC5)
print("params  :", params5)

In [ ]:
data5 = LPNE.prepare_data(cfg5)
splits5, scaler5 = data5['splits'], data5['scaler']

In [ ]:
# ---- initial conditions: the shared GP draws (experimental prep), laid out and scaled for this setup ----
IC5 = emc.make_ic(cfg5, N_IC5, PARAMS, scaler5, splits5, study_dir=STUDY_DIR)   # (N_IC, n_ages, n_features): this setup's states + its own forcing
print("initial-condition records:", IC5.shape)

In [ ]:
# ---- train one model per seed in parallel (N_WORKERS processes; same data, split and settings, only the seed differs) ----
MC5_BEST_EPOCH, MC5_HIST = emc.train_mc(DATASET, CFG5_KW, MC5_SEEDS, MC5_DIR, PARAMS,
                                        n_workers=N_WORKERS, n_threads=N_THREADS, retrain=RETRAIN5)

In [ ]:
# ---- roll every model out from every initial condition -> ROLL5[model, IC] (scaled, states + forcings) ----
w5 = cfg5['window_size']
emu5, ROLL5 = emc.rollout_mc(cfg5, PARAMS, MC5_PATHS, IC5)

full5 = scaler5.transform(splits5['full'])   # ground truth: the GP mean record, scaled
fullt5 = splits5['time']
testt5, traint5 = splits5['test_time'], splits5['train_time']
test_end_5 = (testt5.min() + traint5.max()) / 2   # test | train boundary
MEAN_ROLL5 = ROLL5.mean(axis=1)                  # (n_models, n_ages, n_features): each model's mean rollout over its ICs
rollout_full_5 = MEAN_ROLL5.mean(axis=0)         # ensemble mean, used by the phase space / spectra / comparison cells
print("rollouts:", ROLL5.shape)

In [ ]:
# ---- L2 against the GP mean, per model, and its mean / variance over the models ----
# For each model: the L2 of its mean rollout (over the N_IC5 ICs), whole record and test set, seed window excluded;
# and, for reference, the mean and variance of the L2 of its individual rollouts. MC5_SUMMARY is the mean and
# variance over the models of the model-mean L2 - the numbers in the rollout plot's titles
MC5_TABLE, MC5_SUMMARY = emc.mc_l2(cfg5, full5, ROLL5, MC5_SEEDS)
#display(MC5_SUMMARY.style.format('{:.4f}'))
#display(MC5_TABLE.style.format('{:.4f}'))

In [ ]:
# ---- Monte Carlo rollout plot: every model x IC rollout faint, GP mean dashed (show_model_means=True also draws
# each model's mean rollout over its ICs) ----
emc.plot_mc_rollouts(cfg5, ROLL5, full5, fullt5, MC5_SUMMARY, test_end_5,
                     title=f"setup 5 ({rc.inputs_label(cfg5)})")

In [ ]:
# phase space (X-Y, X-Z, Y-Z) of setup 5's ensemble-mean rollout (mean over all models and ICs) over the ground truth
sp.phase_space(emu5, full5, w5, rollout_data=rollout_full_5)

In [ ]:
# multitaper spectrum of X (ice mass): setup 5's ensemble-mean rollout (mean over all models and ICs) vs. the GPR
# mean ground truth, with the 100 / 41 / 23 kyr orbital periods marked
spectra_X_5 = sp.frequency_spectra(emu5, full5, w5,
                                  timestep=cfg5['timestep'],
                                  spec_var=0,             # 0 = X
                                  rollout_data=rollout_full_5)

### Comp

Comparison of all setups that have been run in this kernel (each setup's rollout cell must have run):
1. table of the L2 and R2 of each state variable, over the whole record and over the test set
2. X's phase relative to the ground truth in the 100 / 41 / 23 kyr bands (lag, phase angle, correlation)
3. X's L2 by setup (X is the one variable every setup has)
4. X rollouts of every setup, stacked on shared axes
5. X's power in the 100 / 41 / 23 kyr orbital bands, relative to the ground truth

In [ ]:
# ---- collect every setup that has been run ----
# setup -> names of its (config, full-record rollout, scaled full record, scaler) in this notebook; the rollout is the
# setup's Monte Carlo ensemble mean (over all models and ICs). Setups whose variables don't exist yet (not run in this
# kernel) are skipped, so the comparison works on partial results. RUNS[s]['mc'] is the setup's per-model L2 table
# (MC<s>_TABLE from emc.mc_l2), used for the spread over the models
SETUP_VARS = {
    '0':  ('cfg0',  'rollout_full_0',  'full0',  'scaler0'),
    '1':  ('cfg1',  'rollout_full_1',  'full1',  'scaler1'),
    '2':  ('cfg2',  'rollout_full_2',  'full2',  'scaler2'),
    '3a': ('cfg3a', 'rollout_full_3a', 'full3a', 'scaler3a'),
    '3b': ('cfg3b', 'rollout_full_3b', 'full3b', 'scaler3b'),
    '3c': ('cfg3c', 'rollout_full_3c', 'full3c', 'scaler3c'),
    '4':  ('cfg4',  'rollout_full_4',  'full4',  'scaler4'),
    '5':  ('cfg5',  'rollout_full_5',  'full5',  'scaler5'),
}
RUNS = {}   # setup -> dict(cfg, rollout, data, scaler, label, mc)
for s, names in SETUP_VARS.items():
    if all(n in globals() for n in names):
        c, ro, data, scl = (globals()[n] for n in names)
        RUNS[s] = dict(cfg=c, rollout=ro, data=data, scaler=scl, label=f"{s}: {rc.inputs_label(c)}",
                       mc=globals().get(f'MC{s}_TABLE'))
print("comparing:", ", ".join(r['label'] for r in RUNS.values()))
print("not run yet:", [s for s in SETUP_VARS if s not in RUNS] or "none")

# ---- L2 and R2 per state variable, overall and over the test set (scaled units, seed window excluded) ----
# Every setup uses the same split and the same X record, so X's scaling - and so its L2 - is comparable across all
# of them; Y and Z only exist in some setups (NaN in the table where a setup doesn't have the variable).
# R2 = 1 - SSE / variance of the ground truth: 1 is perfect, 0 is no better than the record's mean, < 0 is worse
rows = {}
for s, r in RUNS.items():
    c, ro, data = r['cfg'], r['rollout'], r['data']
    w_s, n_test = c['window_size'], c['test_size']
    row = {}
    for k, name in enumerate(rc.state_names(c)):
        full_rows, test_rows = slice(w_s, len(ro)), slice(w_s, n_test)
        row[f'L2 {name}'] = mean_squared_error(data[full_rows, k], ro[full_rows, k])
        row[f'L2 {name} (test)'] = mean_squared_error(data[test_rows, k], ro[test_rows, k])
        row[f'R2 {name}'] = r2_score(data[full_rows, k], ro[full_rows, k])
        row[f'R2 {name} (test)'] = r2_score(data[test_rows, k], ro[test_rows, k])
    rows[r['label']] = row
cols = [f'{m} {v}{t}' for v in ('X', 'Y', 'Z') for m in ('L2', 'R2') for t in ('', ' (test)')]
L2_TABLE = pd.DataFrame.from_dict(rows, orient='index').reindex(columns=cols)   # L2 and R2 (name kept for the cells below)

# shading per column, darker = worse: high L2 is bad, low R2 is bad
l2_cols = [c for c in cols if c.startswith('L2')]
r2_cols = [c for c in cols if c.startswith('R2')]
display(L2_TABLE.style.format('{:.3f}', na_rep='-')
        .background_gradient(cmap='Greys', subset=l2_cols, axis=0)
        .background_gradient(cmap='Greys_r', subset=r2_cols, axis=0))

In [ ]:
# ---- phase of X: does each rollout's timing match the ground truth, band by band? ----
# L2 punishes a rollout that is right in shape but shifted in time as hard as one that is just wrong, so this
# separates the two. X (rollout and ground truth, seed window excluded) is band-passed around each orbital period
# (zero-phase Butterworth, so the filter adds no lag of its own) and the two are cross-correlated:
#   lag   - shift [kyr] at the correlation peak; > 0: the rollout LAGS the ground truth (rows run oldest -> youngest)
#   phase - that lag as an angle of the band's cycle [deg] (360 = one full period; searched within +/- half a period)
#   r     - correlation at the peak (1 = same shape once shifted; low r = the band's signal itself is off)
from scipy import signal

PHASE_BANDS = {'100 kyr': (90, 110), '41 kyr': (38, 44), '23 kyr': (19, 25)}   # band -> (shortest, longest) period

def band_phase(x_emu, x_true, dt, lo, hi):
    """lag [kyr], phase [deg] and peak correlation of x_emu vs. x_true in the period band lo-hi kyr"""
    period = (lo + hi) / 2
    sos = signal.butter(4, [1 / hi, 1 / lo], btype='band', fs=1 / dt, output='sos')
    a, b = signal.sosfiltfilt(sos, x_emu), signal.sosfiltfilt(sos, x_true)
    a, b = a - a.mean(), b - b.mean()
    corr = signal.correlate(a, b, mode='full') / (len(a) * a.std() * b.std())
    lags = signal.correlation_lags(len(a), len(b), mode='full') * dt
    keep = np.abs(lags) <= period / 2
    k = np.argmax(corr[keep])
    lag = lags[keep][k]
    return lag, 360 * lag / period, corr[keep][k]

phase_rows = {}
for s, r in RUNS.items():
    w_s, dt_s, n = r['cfg']['window_size'], r['cfg']['timestep'], r['rollout'].shape[0]
    row = {}
    for band, (lo, hi) in PHASE_BANDS.items():
        lag, ph, rr = band_phase(r['rollout'][w_s:, 0], r['data'][w_s:n, 0], dt_s, lo, hi)
        row[(band, 'lag [kyr]')], row[(band, 'phase [deg]')], row[(band, 'r')] = lag, ph, rr
    phase_rows[r['label']] = row
PHASE_TABLE = pd.DataFrame.from_dict(phase_rows, orient='index')
PHASE_TABLE.columns = pd.MultiIndex.from_tuples(PHASE_TABLE.columns)
display(PHASE_TABLE.style.format('{:.2f}').format('{:+.0f}', subset=[c for c in PHASE_TABLE.columns if c[1] != 'r']))

# heatmap: phase shift of each band (color, 0 = in phase) with the lag and r written in each cell
ph = PHASE_TABLE.xs('phase [deg]', axis=1, level=1)
with plt.rc_context({'font.size': 15}):
    fig, ax = plt.subplots(figsize=(11, 0.7 * len(ph) + 2.5))
    lim = max(30.0, np.nanmax(np.abs(ph.values)))   # symmetric around 0 = in phase
    im = ax.imshow(ph.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
    for (i, j), v in np.ndenumerate(ph.values):
        band = ph.columns[j]
        lag, rr = PHASE_TABLE.iloc[i][(band, 'lag [kyr]')], PHASE_TABLE.iloc[i][(band, 'r')]
        ax.text(j, i, f"{lag:+.0f} kyr\nr = {rr:.2f}", ha='center', va='center', fontsize=12,
                color='white' if abs(v) > 0.6 * lim else 'black')
    ax.set_xticks(range(len(ph.columns)), ph.columns)
    ax.set_yticks(range(len(ph)), ph.index)
    ax.set_title("X phase vs. ground truth, by orbital band", fontweight='bold')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("phase shift [deg]  (> 0: rollout lags)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- X error by setup: overall vs. test-set L2 (X is the one variable every setup has) ----
# bar: mean over the setup's Monte Carlo models of the L2 of each model's mean rollout (over its ICs) - the mu in the
# setup's rollout-plot titles. Error bar: +/- N_SIGMA standard deviations of that L2 over the models (the spread
# between seeds, not the uncertainty of the mean). The lower whisker is cut at 0, since an L2 can't be negative
LOG_Y = False   # True: log y-axis, for when one setup's error dwarfs the others
N_SIGMA = 3
MC_RUNS = {r['label']: r['mc'] for r in RUNS.values() if r['mc'] is not None}
print("no Monte Carlo table (left out):", [r['label'] for r in RUNS.values() if r['mc'] is None] or "none")
labels = list(MC_RUNS)
x_pos = np.arange(len(labels))
width = 0.38
with plt.rc_context({'font.size': 15}):
    fig, ax = plt.subplots(figsize=(18, 6))
    for part, offset, color, name in (('whole', -width / 2, '#1E88E5', 'whole record'),
                                      ('test',  +width / 2, '#D6195E', 'test set only')):
        col = ('X', f'L2 model mean ({part})')
        mu = np.array([MC_RUNS[l][col].mean() for l in labels])
        err = N_SIGMA * np.array([MC_RUNS[l][col].std() for l in labels])   # std over the models (ddof=1, as the var in mc_l2)
        lower = np.minimum(err, mu * (0.99 if LOG_Y else 1.0))            # never below 0 (or to 0 on a log axis)
        ax.bar(x_pos + offset, mu, width, color=color, label=name, yerr=[lower, err], capsize=6,
               error_kw=dict(ecolor='black', elinewidth=1.3, capthick=1.3))
        for xi, (m_, e_) in enumerate(zip(mu, err)):   # values above the whiskers
            ax.text(xi + offset, m_ + e_, f"{m_:.2f}", ha='center', va='bottom', fontsize=12, color='dimgray')
    ax.set_xticks(x_pos, labels, rotation=20, ha='right')
    ax.set_ylabel(r"$\mathcal{L}_{L_2}$ of X (scaled units)")
    if LOG_Y:
        ax.set_yscale('log')
    n_models = sorted({len(t) for t in MC_RUNS.values()})
    ax.set_title(f"X (ice mass) rollout error by ablation setup: mean $\\pm$ {N_SIGMA}$\\sigma$ over "
                 f"{'/'.join(map(str, n_models))} models", fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- X rollout of every setup over the ground truth, one panel per setup, shared axes ----
# physical units [permil] (each setup's own scaler, though X's scaling is the same in all of them)
COLOR_TRUTH, COLOR_EMU = "#D6195E", "#FFB600"   # X's (data, emulator) colors in sp.plot_rollout
ref = next(iter(RUNS.values()))
ages = splits['time'] if 'splits' in globals() else rc.split_data(rc.load_data(ref['cfg']), ref['cfg'])['time']
TEST_END_C = (ages[ref['cfg']['test_size'] - 1] + ages[ref['cfg']['test_size'] + 1]) / 2   # test | train boundary

with plt.rc_context({'font.size': 14}):
    fig, axs = plt.subplots(len(RUNS), 1, figsize=(22, 2.8 * len(RUNS)), sharex=True, sharey=True, squeeze=False)
    axs = axs[:, 0]
    for ax, (s, r) in zip(axs, RUNS.items()):
        to_phys = lambda x: x * r['scaler'].scale_[0] + r['scaler'].mean_[0]   # X column back to d18O [permil]
        n = r['rollout'].shape[0]
        ax.plot(ages[:n], to_phys(r['data'][:n, 0]), color=COLOR_TRUTH, linestyle='dashed', lw=1.8, alpha=0.6,
                label="GPR mean (ground truth)")
        ax.plot(ages[:n], to_phys(r['rollout'][:, 0]), color=COLOR_EMU, lw=1.8, label="emulator rollout")
        ax.axvline(TEST_END_C, color='black', linestyle=':', lw=1.5, label="test | train / validation")
        ax.set_title(f"{r['label']}:   " + r"$\mathcal{L}_{L_2}$" +
                     f" = {L2_TABLE.loc[r['label'], 'L2 X']:.3f}   (test set: {L2_TABLE.loc[r['label'], 'L2 X (test)']:.3f})",
                     fontsize=14, loc='left')
        ax.set_ylabel(r"$\delta^{18}$O [‰]")
        ax.grid(True, alpha=0.3)
    handles, leg_labels = axs[0].get_legend_handles_labels()   # one legend above the panels, clear of the data
    fig.legend(handles, leg_labels, loc='upper center', ncol=3, fontsize=13, frameon=False, bbox_to_anchor=(0.5, 1.0))
    axs[-1].invert_xaxis()   # shared x-axis: oldest on the left
    axs[-1].set_xlabel("time [kyr]")
    plt.tight_layout(rect=[0, 0, 1, 1 - 0.35 / len(RUNS)])
    plt.show()

In [ ]:
# ---- orbital-band power of X: rollout relative to the ground truth ----
# Multitaper spectrum (sp.multitaper_spectrum, as in the spectra plots above) of each setup's X rollout and of the
# ground truth, over the same rows; mean power in each orbital band, as log2(rollout / ground truth):
# 0 = right amount of power, +1 = twice as much, -1 = half. Seed window excluded (those rows ARE the ground truth)
BANDS = {'100 kyr\n(90-110)': (1/110, 1/90), '41 kyr\n(38-44)': (1/44, 1/38), '23 kyr\n(19-25)': (1/25, 1/19)}

def band_power(x, dt):
    freq, spec, *_ = sp.multitaper_spectrum(np.asarray(x, dtype=float), dt)
    return {b: spec[(freq >= lo) & (freq <= hi)].mean() for b, (lo, hi) in BANDS.items()}

ratios = {}
for s, r in RUNS.items():
    w_s, dt_s = r['cfg']['window_size'], r['cfg']['timestep']
    n = r['rollout'].shape[0]
    gt, emu_p = band_power(r['data'][w_s:n, 0], dt_s), band_power(r['rollout'][w_s:, 0], dt_s)
    ratios[r['label']] = {b: np.log2(emu_p[b] / gt[b]) for b in BANDS}
BAND_TABLE = pd.DataFrame.from_dict(ratios, orient='index')

with plt.rc_context({'font.size': 15}):
    fig, ax = plt.subplots(figsize=(11, 0.7 * len(BAND_TABLE) + 2.5))
    lim = max(1.0, np.nanmax(np.abs(BAND_TABLE.values)))   # symmetric color scale around 0 (= matches the data)
    im = ax.imshow(BAND_TABLE.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
    for (i, j), v in np.ndenumerate(BAND_TABLE.values):
        ax.text(j, i, f"{2 ** v:.2f}x", ha='center', va='center', fontsize=13,
                color='white' if abs(v) > 0.6 * lim else 'black')
    ax.set_xticks(range(len(BANDS)), BAND_TABLE.columns)
    ax.set_yticks(range(len(BAND_TABLE)), BAND_TABLE.index)
    ax.set_title("X power in the orbital bands: rollout / ground truth", fontweight='bold')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"log$_2$(rollout / ground truth)  (0 = matches)")
    plt.tight_layout()
    plt.show()

### other

In [ ]:
# do a test where there's one trained model but turn some of the variables into forcings (i.e. tells us which variables' error causes variation in X)
# can look at specific events (i.e. peaks or wupards/downwards moments) to see which variables are dominant for them
# above requires one model for comparabiltiy
# partial teacher forcing --> train model and then force one of the output variables

# should update analysis here in two ways
# MC (IC + seed) rollouts to see error spread on models
# add baseline comparison --> check granger causality

# subsequently:
# permutation importance/SHAP values on one model: look at whole L2 and also error on specific interglacials
# teacher forced tests of models --> how does a single model change when it error in one state variable is removed


In [ ]:
# legacy code, in case something goes wrong
'''
# ---- train one model per seed on the GP mean record (same data, split and settings; only the seed differs) ----
def cfg0_seed(s):
    return rc.make_config(DATASET, seed=int(s), params=PARAMS, train_start=TRAIN_START, insol='insol_ber90',
                          forcing=('insol',))

MC0_PATHS = {s: rc.run_paths(cfg0_seed(s)['run_tag'], MC0_DIR) for s in MC0_SEEDS}
MC0_BEST_EPOCH = {}   # seed -> best physically-valid epoch (None: none passed the ACF gate, last epoch kept)
for s in tqdm(MC0_SEEDS, desc="training"):
    cfg_s, p_s = cfg0_seed(s), MC0_PATHS[s]
    if p_s['weights'].exists() and not RETRAIN0:
        continue
    rc.save_run(cfg_s, scaler0, splits0, runs_dir=MC0_DIR)
    # clears the Keras session and seeds with s first; quiet: 30 models, so keep the per-epoch messages quiet
    emu_s, _, check_s = emc.fit_emulator(PARAMS, data0, cfg_s, cfg_s['acf_threshold'], 15, p_s['weights'],
                                         seed=int(s), quiet=True)
    if p_s['weights'].exists():
        MC0_BEST_EPOCH[s] = check_s.best_epoch
    else:
        print(f"seed {s}: no epoch passed the ACF gate - keeping the last epoch's weights")
        MC0_BEST_EPOCH[s] = None
        emu_s.save_weights(str(p_s['weights']))
    del emu_s, check_s
print("trained this session:", MC0_BEST_EPOCH or "none (all weights already existed)")
'''